In [ ]:
%pip install openpyxl

In [ ]:
# checking the dataset
import pandas as pd
import numpy as np

FILE_PATH = '. Group Project/Post mid-term/cleaned_dataset_200_features.xlsx'

df = pd.read_excel(FILE_PATH, engine="openpyxl")

print("Dataset shape:", df.shape)
display(df.head())

print("=" * 65)
print("DATASET OVERVIEW")
print("=" * 65)

print("Shape (rows, columns):", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:3}. {col}")

print("\nData types:")
print(df.dtypes.value_counts())

print("\nMemory usage (MB):")
print(round(df.memory_usage(deep=True).sum() / 1024**2, 2))

In [ ]:
import numpy as np

FILE_PATH = '. Group Project/Post mid-term/cleaned_dataset_200_features.xlsx'

xls = pd.ExcelFile(FILE_PATH)

for i, sheet in enumerate(xls.sheet_names):
    print(f"Sheet {i}: {sheet}")

In [ ]:
train_df = pd.read_excel(FILE_PATH, sheet_name=2)
test_df = pd.read_excel(FILE_PATH, sheet_name=3)

print("TRAINING DATASET")
print("Shape:", train_df.shape)
display(train_df.head())

print("\nTESTING DATASET")
print("Shape:", test_df.shape)
display(test_df.head())

In [ ]:
train_cols = set(train_df.columns)
test_cols = set(test_df.columns)

print("Training columns:", len(train_cols))
print("Testing columns:", len(test_cols))

print("\nColumns only in training:")
print(train_cols - test_cols)

print("\nColumns only in testing:")
print(test_cols - train_cols)

print("\nCommon columns:", len(train_cols & test_cols))

In [ ]:
print("=" * 60)
print("TRAIN/TEST OVERLAP AUDIT")
print("=" * 60)

# Exact duplicate rows within each dataset
print("Duplicates within training:", train_df.duplicated().sum())
print("Duplicates within testing:", test_df.duplicated().sum())

# Exact matching rows across datasets
common_cols = train_df.columns.intersection(test_df.columns).tolist()

train_unique = train_df[common_cols].drop_duplicates()
test_unique = test_df[common_cols].drop_duplicates()

overlap = train_unique.merge(
    test_unique,
    how="inner",
    on=common_cols
)

print("\nUnique identical rows across train/test:", len(overlap))

In [ ]:
id_candidates = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "patient_id",
    "admission_id"
]

for col in id_candidates:
    if col in train_df.columns and col in test_df.columns:

        train_ids = set(train_df[col].dropna())
        test_ids = set(test_df[col].dropna())

        shared = train_ids.intersection(test_ids)

        print(f"\nIdentifier: {col}")
        print("Training unique IDs:", len(train_ids))
        print("Testing unique IDs:", len(test_ids))
        print("Shared IDs:", len(shared))

if not any(
    col in train_df.columns and col in test_df.columns
    for col in id_candidates
):
    print("WARNING: No shared identifier columns available.")
    print("Cannot establish patient-level independence.")

In [ ]:
TARGET = "icu_transfer"

for name, dataset in [
    ("TRAINING", train_df),
    ("TESTING", test_df)
]:
    print(f"\n{name} DATASET")

    if TARGET in dataset.columns:
        print("Class counts:")
        print(dataset[TARGET].value_counts(dropna=False))

        print("\nClass percentages:")
        print(
            dataset[TARGET]
            .value_counts(normalize=True, dropna=False)
            .mul(100)
            .round(2)
        )
    else:
        print("Target column not found.")

In [ ]:
readme_df = pd.read_excel(
    FILE_PATH,
    sheet_name=1,
    header=None
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)

display(readme_df)

In [ ]:
# prepping dataset

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve
)

TARGET = "icu_transfer"
EXCLUDE = ["subject_id", "hadm_id", TARGET]

feature_cols = [
    c for c in train_df.columns
    if c not in EXCLUDE
]

assert len(feature_cols) == 200
assert set(train_df.columns) == set(test_df.columns)
assert set(train_df["subject_id"]).isdisjoint(
    set(test_df["subject_id"])
)

X = train_df[feature_cols].apply(
    pd.to_numeric, errors="coerce"
)

y = train_df[TARGET].astype(int)

groups = train_df["subject_id"]

print("Feature matrix:", X.shape)
print("Target distribution:")
print(y.value_counts())

In [ ]:
print("train_df exists:", "train_df" in globals())
print("test_df exists:", "test_df" in globals())
print("df exists:", "df" in globals())

In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

print("\nTraining positive rate:", y_train.mean())
print("Validation positive rate:", y_val.mean())

assert set(groups.iloc[train_idx]).isdisjoint(
    set(groups.iloc[val_idx])
)

In [ ]:
preprocessor = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True,
        keep_empty_features=True
    )),
    ("scaler", StandardScaler())
])

In [ ]:
svm_model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LinearSVC(
        C=0.1,
        class_weight="balanced",
        max_iter=10000,
        dual=False,
        random_state=42
    ))
])

svm_model.fit(X_train, y_train)

svm_scores = svm_model.decision_function(X_val)

print("SVM training complete.")

In [ ]:
mlp_model = Pipeline([
    ("preprocess", Pipeline([
        ("imputer", SimpleImputer(
            strategy="median",
            add_indicator=True,
            keep_empty_features=True
        )),
        ("scaler", StandardScaler())
    ])),
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=0.01,
        batch_size=256,
        learning_rate_init=0.001,
        max_iter=100,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=10,
        random_state=42
    ))
])

mlp_model.fit(X_train, y_train)

mlp_scores = mlp_model.predict_proba(X_val)[:, 1]

print("MLP training complete.")

In [ ]:
def evaluate_model(name, y_true, scores, threshold):
    predictions = (scores >= threshold).astype(int)

    return {
        "Model": name,
        "PR-AUC (AP)": average_precision_score(y_true, scores),
        "ROC-AUC": roc_auc_score(y_true, scores),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "Balanced Accuracy": balanced_accuracy_score(
            y_true, predictions
        )
    }

results = pd.DataFrame([
    evaluate_model("Linear SVM", y_val, svm_scores, 0.0),
    evaluate_model("MLP", y_val, mlp_scores, 0.5)
])

display(results.round(4))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

for name, scores in [
    ("Linear SVM", svm_scores),
    ("MLP", mlp_scores)
]:
    precision, recall, thresholds = precision_recall_curve(
        y_val, scores
    )

    ap = average_precision_score(y_val, scores)

    plt.plot(
        recall,
        precision,
        label=f"{name} (AP={ap:.3f})"
    )

plt.axhline(
    y_val.mean(),
    linestyle="--",
    label="No-skill baseline"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Validation Precision-Recall Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
def optimize_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(
        y_true, scores
    )

    # Last PR point has no corresponding threshold
    p = precision[:-1]
    r = recall[:-1]

    f1 = 2 * p * r / np.maximum(p + r, 1e-12)

    best_f1_idx = np.argmax(f1)

    eligible = np.where(r >= 0.80)[0]

    output = {
        "Max F1 threshold": thresholds[best_f1_idx],
        "Max F1": f1[best_f1_idx],
        "Recall at Max F1": r[best_f1_idx],
        "Precision at Max F1": p[best_f1_idx]
    }

    if len(eligible):
        best_recall_idx = eligible[np.argmax(p[eligible])]

        output.update({
            "Recall >=80% threshold": thresholds[best_recall_idx],
            "Precision at >=80% recall": p[best_recall_idx],
            "Recall achieved": r[best_recall_idx]
        })

    return output

for name, scores in [
    ("Linear SVM", svm_scores),
    ("MLP", mlp_scores)
]:
    print(f"\n{name}")
    print(optimize_threshold(y_val, scores))

In [ ]:
# got suspiscious because SVM took 6.5 mins to train and MLP took 10s

# SVM diagnostics
svm = svm_model.named_steps["classifier"]

print("SVM iterations:", svm.n_iter_)
print("SVM max iterations:", svm.max_iter)
print("SVM tolerance:", svm.tol)

# MLP diagnostics
mlp = mlp_model.named_steps["classifier"]

print("\nMLP epochs completed:", mlp.n_iter_)
print("MLP maximum epochs:", mlp.max_iter)
print("MLP final loss:", mlp.loss_)
print("MLP early stopping enabled:", mlp.early_stopping)

print("\nMLP loss curve length:", len(mlp.loss_curve_))

In [ ]:
# 17/10/26

# =====================================================
# DATASET 200
# =====================================================

DATASET_200_PATH = Path(
    "."
    "Final Group Project/Post mid-term/"
    "cleaned_dataset_200_features.xlsx"
)

# Reuse already-loaded DataFrames if available
if "train_df" not in globals():
    train_df = pd.read_excel(
        DATASET_200_PATH,
        sheet_name="Training Data"
    )

if "test_df" not in globals():
    test_df = pd.read_excel(
        DATASET_200_PATH,
        sheet_name="Testing Data"
    )

# Verify same admission population and split
assert set(train_df["hadm_id"]) == set(train_75["hadm_id"])
assert set(test_df["hadm_id"]) == set(test_75["hadm_id"])

# Verify identical target labels
for a, b in [
    (train_df, train_75),
    (test_df, test_75)
]:
    check = a[["hadm_id", TARGET]].merge(
        b[["hadm_id", TARGET]],
        on="hadm_id",
        suffixes=("_200", "_75"),
        validate="one_to_one"
    )

    assert (
        check[f"{TARGET}_200"] ==
        check[f"{TARGET}_75"]
    ).all(), "Target labels differ!"

# Run the models
results_200 = run_models(
    dataset_name="200_Features",
    train_data=train_df,
    test_data=test_df,
    validation_subject_ids=validation_subject_ids
)

display(results_200.round(4))